# Lab 3 · Repaired manual ReAct foundation

## Goal
Trace a bounded model/tool exchange and test dispatch independently of a provider.

## Setup
Use the Labs 3–8 environment. Implement the tasks below, set RUN_EXERCISES=True, restart and run all. Default execution does not solve or grade the assignment. All fixtures are invented. Instructor solutions are distributed separately.

In [ ]:
RUN_EXERCISES = True

## Provided support
These utilities support the assignment; the agent/retrieval logic is your work.

In [ ]:
from __future__ import annotations

import ast

import math

import operator

import re

from dataclasses import dataclass, field

from heapq import nlargest

_OPS={ast.Add:operator.add,ast.Sub:operator.sub,ast.Mult:operator.mul,ast.Div:operator.truediv,
      ast.FloorDiv:operator.floordiv,ast.Mod:operator.mod,ast.Pow:operator.pow}

def calculator(expr: str) -> str:
    """Bounded arithmetic with at most 256 characters, 64 AST nodes, and magnitude 1e12."""
    def finite(value):
        if type(value) not in (int,float) or abs(value)>1e12 or not math.isfinite(value):
            raise ValueError("numeric limit")
        return value
    def compute(node,depth=0):
        if depth>16: raise ValueError("nesting limit")
        if isinstance(node,ast.Expression):return compute(node.body,depth+1)
        if isinstance(node,ast.Constant):return finite(node.value)
        if isinstance(node,ast.UnaryOp) and type(node.op) in (ast.UAdd,ast.USub):
            return finite(compute(node.operand,depth+1)*(1 if isinstance(node.op,ast.UAdd) else -1))
        if isinstance(node,ast.BinOp) and type(node.op) in _OPS:
            a,b=compute(node.left,depth+1),compute(node.right,depth+1)
            if isinstance(node.op,ast.Pow) and (type(b) is not int or abs(b)>12):raise ValueError("exponent limit")
            return finite(_OPS[type(node.op)](a,b))
        raise ValueError("unsupported expression")
    if not isinstance(expr,str) or not expr.strip() or len(expr)>256:return "calculator error: invalid input"
    try:
        tree=ast.parse(expr,mode="eval")
        if sum(1 for _ in ast.walk(tree))>64:raise ValueError("expression limit")
        return str(compute(tree))
    except (ValueError,SyntaxError,ArithmeticError,RecursionError):
        return "calculator error: invalid or out-of-bounds expression"

def word_count(text:str)->str:
    if not isinstance(text,str) or len(text)>10_000:raise ValueError("text limit")
    return str(len(text.split()))

TOOLS={"calculator":(calculator,"bounded arithmetic"),"word_count":(word_count,"count whitespace-separated words")}

@dataclass
class Result:
    answer: str|None
    steps: list=field(default_factory=list)
    stopped_reason: str="final_answer"

### TODO: parse
Parse one Action: name[input] or Final Answer: text. Optionally allow a single leading Thought line as an observable protocol label, not a claim about hidden reasoning. Reject fabricated Observation lines, multiple actions, nonstrings, and messages longer than 10,000 characters. Return ("final", text), ("action", name, input), or ("error", None).

In [ ]:
def parse(text):
    if not isinstance(text,str) or len(text)>10_000:return ("error",None)
    # One action or final result only. A supplied Observation is a protocol violation.
    if re.search(r"(?m)^Observation:",text):return ("error",None)
    body=re.sub(r"\AThought:[^\n]*\n", "",text.strip()).strip()
    match=re.fullmatch(r"Final Answer:\s*(.+)",body,re.S)
    if match:return ("final",match.group(1).strip())
    match=re.fullmatch(r"Action:\s*([A-Za-z_]\w*)\[([^\[\]]*)\]",body,re.S)
    return ("action",match.group(1),match.group(2)) if match else ("error",None)

### TODO: run_tool
Use only the supplied registry. Bound string input at 10,000 characters, reject unknown tools, and return sanitized errors. Never execute a rejected request.

In [ ]:
def run_tool(action,action_input,tools=TOOLS):
    if not isinstance(action,str) or action not in tools:return "tool error: unknown tool"
    if not isinstance(action_input,str) or len(action_input)>10_000:return "tool error: invalid input"
    try:return tools[action][0](action_input)
    except Exception:return "tool error: execution failed"

### TODO: run
Implement the model/tool loop. Validate a nonempty task of at most 10,000 characters and an integer max_steps in 1..32. Forward the Observation stop sequence, append actual observations to history, retain action traces, and return Result with final_answer, parse_error, model_error, or max_steps.

In [ ]:
def run(task,llm,tools=TOOLS,max_steps=8):
    if type(max_steps) is not int or not 1<=max_steps<=32:raise ValueError("step budget must be 1..32")
    if not isinstance(task,str) or not task.strip() or len(task)>10_000:raise ValueError("invalid task")
    menu="\n".join(f"{name}: {desc}" for name,(_,desc) in tools.items())
    messages=[{"role":"system","content":"Emit Action: name[input] or Final Answer: text. Never emit Observation.\n"+menu},{"role":"user","content":task}]
    steps=[]
    for _ in range(max_steps):
        try:reply=llm(messages,stop=["Observation:"])
        except Exception:return Result(None,steps,"model_error")
        parsed=parse(reply)
        if parsed[0]=="final":return Result(parsed[1],steps)
        if parsed[0]=="error":return Result(None,steps,"parse_error")
        _,name,args=parsed
        observation=run_tool(name,args,tools)
        steps.append({"action":name,"input":args,"observation":observation})
        messages.extend([{"role":"assistant","content":reply},{"role":"user","content":"Observation: "+str(observation)}])
    return Result(None,steps,"max_steps")

## Checks
Visible checks are examples, not a complete grader. Add two normal and three failure cases.

In [ ]:
if RUN_EXERCISES:
    calls=[]
    def fake(messages,stop):
        assert stop==["Observation:"]
        calls.append(messages[-1]["content"])
        if len(calls)==1:return "Action: calculator[2*21]"
        assert messages[-1]["content"]=="Observation: 42"
        return "Final Answer: 42"
    result=run("Calculate 2*21",fake)
    assert result.answer=="42"
    assert run_tool("calculator","",{"calculator":(lambda _:"replacement","")})=="replacement"
    print(result)

In [ ]:
if RUN_EXERCISES:
    assert calculator("2*(3+4)")=="14"
    for bad in ["2**1000000000","True","__import__('os')", "1/0", "1e999", "9"*257]:
        assert calculator(bad).startswith("calculator error")
    assert parse("Action: missing[x]\nFinal Answer: fake")[0]=="error"
    assert run_tool("missing","x")=="tool error: unknown tool"
    assert run("task",lambda *a,**kw:"bad").stopped_reason=="parse_error"
    assert run("task",lambda *a,**kw:"Action: word_count[a b]",max_steps=1).stopped_reason=="max_steps"
    print("Foundation checks passed")

## Submission and rubric
Submit your implementation, test evidence, and a trace explanation. Implementation 50 points, added failure tests 25, trace and limitations 25. A default run with exercises disabled does not pass the lab. Do not import reference_support or instructor solutions to implement the tasks.